<a href="https://colab.research.google.com/github/nitinth001/cartpole-rl/blob/main/sarsa_cartpole.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import gymnasium as gym
import tensorflow as tf
from keras import Model, Input
from keras.layers import Dense
import numpy as np

env = gym.make("CartPole-v1")

# Q Network
net_input = Input(shape=(4,))
x = Dense(64, activation="relu")(net_input)
x = Dense(32, activation="relu")(x)
output = Dense(2, activation="linear")(x)

q_net = Model(inputs=net_input, outputs=output)

# Parameters
ALPHA = 0.001
EPSILON = 1.0
EPSILON_DECAY = 1.001
GAMMA = 0.99
NUM_EPISODES = 500


# Policy (epsilon-greedy)
def policy(state, explore=0.0):

    q_values = q_net(state)
    action = tf.argmax(q_values[0]).numpy()

    if np.random.random() <= explore:
        action = np.random.randint(0, 2)

    return action


for episode in range(NUM_EPISODES):

    done = False
    total_reward = 0
    episode_length = 0

    state, info = env.reset()
    state = tf.convert_to_tensor([state], dtype=tf.float32)

    action = policy(state, EPSILON)

    while not done:

        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        next_state = tf.convert_to_tensor([next_state], dtype=tf.float32)

        next_action = policy(next_state, EPSILON)

        target = reward + GAMMA * q_net(next_state)[0][next_action]

        if done:
            target = reward

        with tf.GradientTape() as tape:
            current = q_net(state)

        grads = tape.gradient(current, q_net.trainable_weights)

        delta = target - current[0][action]

        for j in range(len(grads)):
            q_net.trainable_weights[j].assign_add(ALPHA * delta * grads[j])

        state = next_state
        action = next_action

        total_reward += reward
        episode_length += 1

    print("Episode:", episode,
          "Length:", episode_length,
          "Reward:", total_reward,
          "Epsilon:", EPSILON)

    EPSILON /= EPSILON_DECAY

q_net.save("sarsa_q_net")

env.close()

Episode: 0 Length: 18 Reward: 18.0 Epsilon: 1.0
Episode: 1 Length: 27 Reward: 27.0 Epsilon: 0.9990009990009991
Episode: 2 Length: 40 Reward: 40.0 Epsilon: 0.9980029960049943
Episode: 3 Length: 12 Reward: 12.0 Epsilon: 0.9970059900149795
Episode: 4 Length: 13 Reward: 13.0 Epsilon: 0.9960099800349447
Episode: 5 Length: 24 Reward: 24.0 Epsilon: 0.9950149650698749
Episode: 6 Length: 11 Reward: 11.0 Epsilon: 0.9940209441257493
Episode: 7 Length: 53 Reward: 53.0 Epsilon: 0.9930279162095398
Episode: 8 Length: 22 Reward: 22.0 Epsilon: 0.9920358803292106
Episode: 9 Length: 13 Reward: 13.0 Epsilon: 0.991044835493717
Episode: 10 Length: 27 Reward: 27.0 Epsilon: 0.9900547807130041
Episode: 11 Length: 25 Reward: 25.0 Epsilon: 0.9890657149980062
Episode: 12 Length: 15 Reward: 15.0 Epsilon: 0.9880776373606457
Episode: 13 Length: 37 Reward: 37.0 Epsilon: 0.987090546813832
Episode: 14 Length: 11 Reward: 11.0 Epsilon: 0.9861044423714607
Episode: 15 Length: 18 Reward: 18.0 Epsilon: 0.9851193230484123
Epi

KeyboardInterrupt: 